# Build the training dataset

Downloads the source datasets, crops one face per image or video frame with RetinaFace on the GPU, and uploads the crops as a **private** Hugging Face dataset in the format `train_decision_model.ipynb` reads.

Every crop is a 256x256 JPEG around one face with 25% margin, whatever its class, so resolution and compression carry no hint of the label. Images with no face, or with two faces of similar size, are dropped and counted per generator.

**Runtime:** Colab Pro with an L4 or A100 GPU.

**Before running:** add a Colab secret named `HF_TOKEN` (key icon on the left) holding a Hugging Face token with write access. The upload cell is the only one that needs it.

**Sources and licenses:** [OpenRL/DeepFakeFace](https://huggingface.co/datasets/OpenRL/DeepFakeFace) (OpenRAIL) and [AxonData/face-anti-spoofing-dataset](https://huggingface.co/datasets/AxonData/face-anti-spoofing-dataset) (CC BY-NC 4.0). The output shows real people's faces: keep the repository private.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"   # created private on upload
BUILD_DIR = "/content/build"
FRAMES_PER_VIDEO = 16
# None processes every image. A number caps images per DeepFakeFace generator, for a quicker build.
MAX_IMAGES_PER_GENERATOR = None
FLUSH_EVERY = 2000

### Installation

RetinaFace runs on TensorFlow, PyTorch or ONNX Runtime. Only TensorFlow and `onnxruntime-gpu` use the GPU, so this cell installs `onnxruntime-gpu` and pins the ONNX backend.

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}"
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

### Download the sources

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download
from face_decision_lab.sources import axondata, deepfakeface

dff_zips = {
    generator: Path(hf_hub_download(deepfakeface.REPO_ID, f"{generator}.zip", repo_type = "dataset"))
    for generator in deepfakeface.GENERATORS
}
axon_root = Path(snapshot_download(axondata.REPO_ID, repo_type = "dataset"))
print({g: f"{p.stat().st_size / 1e9:.2f} GB" for g, p in dff_zips.items()})
print("AxonData files:", sum(1 for p in axon_root.rglob("*") if p.is_file()))

### Measure detection speed

Times RetinaFace on the first 100 DeepFakeFace images before committing to the full build.

In [ ]:
import time, zipfile
from face_decision_lab.build import decode_image
from face_decision_lab.faces import RetinaFaceDetector

detector = RetinaFaceDetector()
with zipfile.ZipFile(dff_zips["wiki"]) as archive:
    names = [n for n in archive.namelist() if n.endswith(".jpg")][:100]
    images = [decode_image(archive.read(n)) for n in names]
detector(images[0])  # warm-up
started = time.perf_counter()
for image in images:
    detector(image)
per_image = (time.perf_counter() - started) / len(images)
print(f"{per_image * 1000:.0f} ms per image; the full DeepFakeFace build (120,000 images) takes about {per_image * 120_000 / 3600:.1f} h of detection")

### Build

Resumable: if the session dies, run the notebook again and it skips every file already handled (`processed.jsonl`).

In [ ]:
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder, folder_files, zip_members

builder = DatasetBuilder(BUILD_DIR, detector, frames_per_video = FRAMES_PER_VIDEO)

for generator, archive in dff_zips.items():
    for count, (item, data) in enumerate(tqdm(zip_members(archive, deepfakeface.parse_member), desc = generator, total = 30_000)):
        if MAX_IMAGES_PER_GENERATOR is not None and count >= MAX_IMAGES_PER_GENERATOR:
            break
        builder.add_image(item, data)
        if count % FLUSH_EVERY == 0:
            builder.flush()

for item, path in tqdm(list(folder_files(axon_root, axondata.parse_path)), desc = "axondata"):
    if item.is_video:
        builder.add_video(item, path)
    else:
        builder.add_image(item, path.read_bytes())
builder.flush()
print(len(builder.samples), "crops")

### Check detection outcomes

A detector that drops one generator much more than another biases the dataset. Review the saved share per generator before uploading.

In [ ]:
for generator in sorted(builder.report.counts):
    counts = builder.report.counts[generator]
    print(f"{generator:24s} saved {builder.report.saved_share(generator):6.1%}  {counts}")

### Upload as a private dataset

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from face_decision_lab.hub import push_build

login(token = userdata.get("HF_TOKEN"))
print(push_build(BUILD_DIR, HF_DATASET))